# 24 · School places (supply) and children (demand) per level — Part D, step D5 (schools, part 1)

**Why:** E2SFCA needs, for every school, how many **places** it offers at each level, and for every 100 m cell,
how many **children** of that level's ages would use Government + Aided schools.

**Supply — two versions per school and level** (`scripts/bigdata/school_capacity.py`):
- **Main run, norm-based:** places = teachers × pupils allowed per teacher by the official rule.
  - **Kerala** (Kerala Education Rules, 5th ed., corrected to 31-07-2026):
    - LP (Std 1–4): LP teachers × **30** (KER Rule 23A → RTE Act 2009 Schedule);
    - UP (Std 5–7): UP teachers × **30** for Std 5 and **35** for Std 6–7 (Rule 23A);
    - HS (Std 8–10): High School Assistants × **35** for Std 8 (Rule 23A) and **45 per division** for Std 9–10
      (Rule 23), one HSA post per division (KER Ch. XXIII Rule 4);
    - HSS: (HSCAP public batches × **50** + VHSCAP batches × **30**) × 2 years (G.O.(Rt) 5293/2026/GEDN; VHSE
      prospectus 2026 s. 21). Sensitivity: 60 seats per HSS batch (marginal increase).
    - A school with pupils at a level but no class teacher listed for it shares all its class teachers between
      its levels by pupils (`split_by_pupils`).
  - **India** (UDISE 2022): the school's teachers (else classrooms: RTE, one classroom per teacher) shared
    between the levels it teaches by pupils × **30 / 35 / 30 / 30** (RTE Schedule; RMSA 30:1; classes 11–12 30:1
    assumed, no separate national norm found).
- **Sensitivity run:** places = pupils enrolled at that level (VHSE: seats, no pupil counts are published).

**Catchment (distance norm):** LP / primary 1 km, UP / upper primary 3 km (RTE Model Rules 2010); HS / secondary 5 km,
3 km in hilly areas (Sametham flag, Kerala HS only); HSS 7 km (Samagra Shiksha).

**Demand share:** children of each level's ages from WorldPop × the share of pupils in Government + Aided
schools, per state and level, from UDISE 2022 (all managements and boards). The Sametham class-wise table was
not used for the share: it counts state-syllabus pupils only (no CBSE/ICSE) and overstated it (84 % vs 61 % for LP).

**Outputs:** `silver/school_supply_kerala`, `silver/school_supply_india` (by state), `silver/school_demand_share`.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Settings

In [3]:
import sys
import pandas as pd

sys.path.insert(0, str(PROJECT / "scripts" / "bigdata"))
import school_capacity as sc

SILVER = PROJECT / "data" / "lake" / "silver"
UDISE = PROJECT / "data" / "raw" / "udise_basic_details" / "basic-details-of-schools.csv"

## Kerala: places per school and level

In [4]:
# Kerala: Sametham levels, KER rules (see the module docstring); main = norm-based, sensitivity = enrolment
k = pd.read_parquet(SILVER / "schools_kerala")
ks = sc.kerala_supply(k)
g = ks.groupby("level").agg(schools=("school_id", "size"), places_norm=("cap_main", "sum"), pupils=("cap_enrol", "sum"))
g["norm_places_per_pupil"] = (g["places_norm"] / g["pupils"]).round(2)
print(g.loc[["lp", "up", "hs", "hss"]].round({"places_norm": 0, "pupils": 0}).to_string())
print("\nhow the norm-based places were found:")
print(ks.groupby(["level", "cap_source"]).size().to_string())
flag = ks.loc[ks["level"].eq("hs"), "school_id"].map(k.set_index("school_code_num")["sam_is_hilly"])
print("\nHS schools by Sametham 'Is Hilly Area' (True = 3 km catchment; empty '--' = treated as plains, 5 km):",
      flag.astype(str).value_counts().to_dict())
print("HSS places with 60 seats per batch (sensitivity):", f"{ks['cap_hss60'].sum():,.0f}")
assert ks["school_id"].isin(k["school_code_num"]).all() and ks[["lat", "lon"]].notna().all().all()
ks.to_parquet(SILVER / "school_supply_kerala", index=False)
print("written silver/school_supply_kerala:", len(ks), "school-level rows")

       schools  places_norm     pupils  norm_places_per_pupil
level                                                        
lp        9392    1316584.0   960557.0                   1.37
up        5696     951740.0   928349.0                   1.03
hs        2714    1506775.0  1141074.0                   1.32
hss       1899     685180.0   792658.0                   0.86

how the norm-based places were found:
level  cap_source          
hs     split_by_pupils           99
       teachers_by_level       2615
hss    hscap_vhscap_batches    1899
lp     split_by_pupils          917
       teachers_by_level       8475
up     split_by_pupils          844
       teachers_by_level       4852

HS schools by Sametham 'Is Hilly Area' (True = 3 km catchment; empty '--' = treated as plains, 5 km): {'False': 1440, 'True': 303}
HSS places with 60 seats per batch (sensitivity): 809,220
written silver/school_supply_kerala: 19701 school-level rows


## India: places per school and level

In [5]:
# India: national levels, UDISE teachers (else classrooms) shared between the levels a school teaches by pupils
s = pd.read_parquet(SILVER / "schools_india")
ins = sc.india_supply(s)
g = ins.groupby("level").agg(schools=("school_id", "size"), places_norm=("cap_main", "sum"), pupils=("cap_enrol", "sum"))
g["norm_places_per_pupil"] = (g["places_norm"] / g["pupils"]).round(2)
print(g.loc[list(sc.INDIA_LEVELS)].round({"places_norm": 0, "pupils": 0}).to_string())
print("\nstaff count used per school:", ins.drop_duplicates("school_id")["cap_source"].value_counts().to_dict(),
      "(enrolment = no teachers and no classrooms recorded)")
print("schools without a location (left out):", int(s["lat"].isna().sum()))
import shutil
shutil.rmtree(SILVER / "school_supply_india", ignore_errors=True)  # a partitioned write adds files: clear old runs first
ins.to_parquet(SILVER / "school_supply_india", index=False, partition_cols=["state_name"])
assert len(pd.read_parquet(SILVER / "school_supply_india")) == len(ins), "supply table does not read back 1:1"
print("written silver/school_supply_india:", len(ins), "school-level rows")

                  schools  places_norm      pupils  norm_places_per_pupil
level                                                                    
primary            908985   85460686.0  71754393.0                   1.19
upper_primary      403895   50900587.0  41228352.0                   1.23
secondary          144301   22909818.0  23641440.0                   0.97
higher_secondary    65427   12748925.0  14800014.0                   0.86



staff count used per school: {'teachers': 1060562, 'classrooms': 8700, 'enrolment': 246} (enrolment = no teachers and no classrooms recorded)
schools without a location (left out): 1


written silver/school_supply_india: 1522608 school-level rows


## Demand: Government + Aided share of children

In [6]:
# Demand: share of children who attend Government + Aided schools (UDISE 2022, all managements and boards)
india = sc.gov_aided_share(UDISE, {lv: (a, b) for lv, (a, b, _) in sc.INDIA_LEVELS.items()}).assign(scope="india")
kerala = sc.gov_aided_share(UDISE, sc.KERALA_CLASSES).query("state_name == 'Kerala'").assign(scope="kerala")
share = pd.concat([india, kerala], ignore_index=True)
assert share["share"].between(0, 1).all() and set(ins["state_name"]) <= set(india["state_name"]), "a state has no share"
print("states with a share:", india["state_name"].nunique(), "(UDISE 2022 state list)")
print("Kerala levels:", kerala.set_index("level")["share"].round(3).to_dict())
print("India, all states:", india.groupby("level")[["pupils_ga", "pupils_all"]].sum().eval("pupils_ga / pupils_all").round(3).to_dict())
print("lowest and highest state shares:")
print(india.sort_values("share").iloc[[0, 1, 2, -3, -2, -1]][["state_name", "level", "share"]].round(3).to_string(index=False))
share.to_parquet(SILVER / "school_demand_share", index=False)
print("written silver/school_demand_share")

states with a share: 35 (UDISE 2022 state list)
Kerala levels: {'lp': 0.611, 'up': 0.71, 'hs': 0.801, 'hss': 0.875}
India, all states: {'higher_secondary': 0.631, 'primary': 0.651, 'secondary': 0.669, 'upper_primary': 0.687}
lowest and highest state shares:
 state_name            level  share
    Manipur        secondary  0.287
    Manipur higher_secondary  0.289
    Manipur    upper_primary  0.291
Lakshadweep        secondary  1.000
Lakshadweep          primary  1.000
Lakshadweep    upper_primary  1.000
written silver/school_demand_share


## Check: modelled children vs enrolled pupils (Kerala)

In [7]:
# Check the demand side on Kerala: WorldPop children x share should be close to the Government + Aided pupils
cells = pd.read_parquet(SILVER / "population_cells", filters=[("state", "=", "Kerala")], columns=["pop_5_9", "pop_10_14", "pop_15_19"])
band = cells.sum()
kids = {"lp": .8 * band["pop_5_9"], "up": .6 * band["pop_10_14"], "hs": .4 * band["pop_10_14"] + .2 * band["pop_15_19"],
        "hss": .4 * band["pop_15_19"]}  # = sc.demand_sql(level)
sh = kerala.set_index("level")["share"]
pupils = ks.groupby("level")["cap_enrol"].sum()
chk = pd.DataFrame({"children_worldpop_M": pd.Series(kids) / 1e6, "share": sh,
                    "modelled_GA_children_M": pd.Series(kids) * sh / 1e6, "GA_pupils_2026_M": pupils / 1e6})
chk["pupils / modelled"] = chk["GA_pupils_2026_M"] / chk["modelled_GA_children_M"]
print(chk.loc[["lp", "up", "hs", "hss"]].round(3).to_string())
assert chk["pupils / modelled"].between(0.9, 1.1).all(), "demand differs from enrolment by more than 10 %"

     children_worldpop_M  share  modelled_GA_children_M  GA_pupils_2026_M  pupils / modelled
lp                 1.626  0.611                   0.993             0.961              0.967
up                 1.353  0.710                   0.960             0.928              0.967
hs                 1.378  0.801                   1.104             1.141              1.033
hss                0.953  0.875                   0.834             0.793              0.951
